# 第12回：改善実験を小さく回す

**今日の問い：改善した理由を後から説明できる実験とは何か。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

- 変更を1要素に限定した比較を設計する
- 交差検証の平均とばらつきを記録する
- 検証データ上の重要度と誤りから次の仮説を選ぶ

### 進み方

`CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
`SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

### 先に押さえる言葉

- 実験ログ：変更・条件・結果・解釈を残す記録
- ハイパーパラメータ：学習前に人が決める設定
- permutation importance：列を崩したときの性能低下で寄与を見る方法
- 再現性：同じ手順で同じ結果を得られる性質

> **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd

df = pd.read_csv(DATA / "compound_experiments.csv")
print(f"{len(df)}行 × {len(df.columns)}列")
df.head()


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

features = ["temperature_c", "reaction_time_h", "concentration_m", "molecular_weight", "logp", "tpsa"]
X, y = df[features], df["active"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


## TRY：1要素だけ変えて記録する


In [ ]:
rows=[]
for depth in [3, 6, None]:
    model=make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=depth, random_state=42))
    scores=cross_validate(model, X, y, cv=cv, scoring="f1", return_train_score=True)
    rows.append({"実験名": f"depth={depth}", "変更点": "max_depthのみ", "学習F1": scores["train_score"].mean(), "検証F1平均": scores["test_score"].mean(), "検証F1標準偏差": scores["test_score"].std()})
experiment_log=pd.DataFrame(rows)
experiment_log.round(3)


## 重要度から次の仮説を考える


In [ ]:
best=make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)).fit(X, y)
importance=permutation_importance(best, X, y, scoring="f1", n_repeats=10, random_state=42)
pd.DataFrame({"特徴量": features, "重要度": importance.importances_mean}).sort_values("重要度", ascending=False).round(3)


## 実験ログの最小項目

- 実験名
- 変えたもの（1つ）
- 固定した比較条件
- 結果の平均とばらつき
- 気づき
- 次の仮説

Copilotには案を出してもらい、優先順位と予測時点の妥当性は人が判断します。


## DEEP DIVE：結果を一段深く読む

次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

### 出力を見る観点

- 標準偏差が改善幅より大きくないか確認する
- 重要度は因果効果ではない
- 仮説は次の実験で反証可能な形にする


In [ ]:
from sklearn.model_selection import train_test_split
X_fit, X_holdout, y_fit, y_holdout = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
best = make_pipeline(SimpleImputer(strategy="median"), RandomForestClassifier(n_estimators=150, max_depth=6, random_state=42)).fit(X_fit, y_fit)
importance = permutation_importance(best, X_holdout, y_holdout, scoring="f1", n_repeats=10, random_state=42)
holdout_importance = pd.DataFrame({"特徴量": features, "重要度平均": importance.importances_mean, "重要度標準偏差": importance.importances_std})
display(holdout_importance.sort_values("重要度平均", ascending=False).round(3))
experiment_log.to_csv(ROOT / "workspace" / "experiment_log.csv", index=False)
print("実験ログをworkspace/experiment_log.csvへ保存しました")


## よくある誤り

- 同時に複数要素を変える
- 学習データ上の重要度だけを見る
- 悪化した実験を記録から消す

## SELF-STUDY（任意・30〜60分）

- 実験ログをCSVへ保存し再読込する
- 重要度上位1列を外すアブレーションを行う

成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

## 振り返りチェック

1. 1要素だけ変える理由は何か
2. 平均と標準偏差をどう読むか
3. 重要度から断定できないことは何か

答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
